In [ ]:
%%time
### cell 0 ###

import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 0 ###

# Optimized code
# 1) Filter and select once, 2) Compute intermediates with minimal copies, 3) Reduce groupby passes by summing and counting, then compute means

date = pd.Timestamp("1998-09-02")
# Step 1: filter and select necessary columns in one go
mask = lineitem.L_SHIPDATE <= date
cols = [
    "L_RETURNFLAG", "L_LINESTATUS", "L_QUANTITY",
    "L_EXTENDEDPRICE", "L_DISCOUNT", "L_TAX", "L_ORDERKEY",
]
lf = lineitem.loc[mask, cols].copy()  # .copy() so we can assign new columns without warnings

In [ ]:
%%time
### cell 1 ###

# Step 2: compute DISC_PRICE and CHARGE once each
lf["DISC_PRICE"] = lf.L_EXTENDEDPRICE * (1 - lf.L_DISCOUNT)
lf["CHARGE"]     = lf.DISC_PRICE * (1 + lf.L_TAX)

# Step 3: perform grouped sums and counts (6 passes instead of 8)
grp = lf.groupby(["L_RETURNFLAG", "L_LINESTATUS"], as_index=False).agg(
    L_QUANTITY_sum      = ("L_QUANTITY",      "sum"),
    L_EXTENDEDPRICE_sum = ("L_EXTENDEDPRICE", "sum"),
    DISC_PRICE          = ("DISC_PRICE",       "sum"),
    CHARGE              = ("CHARGE",           "sum"),
    L_ORDERKEY          = ("L_ORDERKEY",       "count"),
    L_DISCOUNT_sum      = ("L_DISCOUNT",       "sum"),
)

# Step 4: compute averages from sums and counts
grp["AVG_QTY"]     = grp.L_QUANTITY_sum      / grp.L_ORDERKEY
grp["AVG_PRICE"]   = grp.L_EXTENDEDPRICE_sum / grp.L_ORDERKEY
grp["L_DISCOUNT"]  = grp.L_DISCOUNT_sum      / grp.L_ORDERKEY

# Step 5: rename summed columns and reorder to match original output
total = grp.rename(
    columns={
        "L_QUANTITY_sum":      "L_QUANTITY",
        "L_EXTENDEDPRICE_sum": "L_EXTENDEDPRICE",
    }
)[
    [
        "L_RETURNFLAG", "L_LINESTATUS", "L_QUANTITY", "L_EXTENDEDPRICE",
        "DISC_PRICE", "CHARGE", "AVG_QTY", "AVG_PRICE", "L_DISCOUNT", "L_ORDERKEY",
    ]
]